In [5]:
import os
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from utils import parse_tophit_clean

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
current_year = datetime.now().year
recent_years = list(range(2025, current_year + 1))

available_charts = {
    "apple": recent_years,
    "itunes": recent_years, 
    "radio": list(range(2003, current_year + 1)),
    "shazam": recent_years,
    "vk": recent_years, 
    "yandex": recent_years, 
    "youtube": list(range(2018, current_year + 1)),
    "zvuk": recent_years
}

output_dir = "tophit_charts"
os.makedirs(output_dir, exist_ok=True)

opts = Options()
opts.add_argument("--headless") 
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
all_data = []

try:
    for platform, years in available_charts.items():
        for year in years:
            url = f"https://tophit.ru/chart/top/{platform}/hits/ru/annual/{year}"
            print(f"Processing: {platform.upper()} {year}")
            
            all_data.extend(parse_tophit_clean(driver, url, year, platform))
finally:
    driver.quit()

df_master = pd.DataFrame(all_data)

Processing: APPLE 2025
Processing: APPLE 2026
Skipping APPLE 2026: Redirected to https://tophit.ru/ru/chart/top/apple/hits/ru/annual
Processing: ITUNES 2025
Processing: ITUNES 2026
Skipping ITUNES 2026: Redirected to https://tophit.ru/ru/chart/top/itunes/hits/ru/annual
Processing: RADIO 2003
Processing: RADIO 2004
Processing: RADIO 2005
Processing: RADIO 2006
Processing: RADIO 2007
Processing: RADIO 2008
Processing: RADIO 2009
Processing: RADIO 2010
Processing: RADIO 2011
Processing: RADIO 2012
Processing: RADIO 2013
Processing: RADIO 2014
Processing: RADIO 2015
Processing: RADIO 2016
Processing: RADIO 2017
Processing: RADIO 2018
Processing: RADIO 2019
Processing: RADIO 2020
Processing: RADIO 2021
Processing: RADIO 2022
Processing: RADIO 2023
Processing: RADIO 2024
Processing: RADIO 2025
Processing: RADIO 2026
Skipping RADIO 2026: Redirected to https://tophit.ru/ru/chart/top/radio/hits/ru/annual
Processing: SHAZAM 2025
Processing: SHAZAM 2026
Skipping SHAZAM 2026: Redirected to https:/

In [ ]:
if not df_master.empty:
    df_master.columns = df_master.columns.str.lower()
    
    if 'release_date' in df_master.columns:
        df_master['release_date'] = pd.to_datetime(df_master['release_date'], errors='coerce').dt.strftime('%Y-%m-%d')
        
    target_cols = ['year', 'platform', 'artist_name', 'song_name', 'release_date', 'language']
    for col in target_cols:
        if col not in df_master.columns:
            df_master[col] = 'russian' if col == 'language' else None
            
    df_master = df_master[target_cols]
    
    master_csv_path = os.path.join(output_dir, "tophit_master_all.csv")
    df_master.to_csv(master_csv_path, index=False, encoding='utf-8')
    print(f"\nFinished! Master DataFrame saved to '{master_csv_path}' with {len(df_master)} rows.")


Finished! Master DataFrame saved to 'tophit_charts/tophit_master_all.csv' with 7389 rows.


In [7]:
df_master.sample(10)

,year,platform,artist_name,song_name,release_date,language
5103,2025,shazam,Dabbackwood,Не попался,NaN,russian
5620,2018,youtube,Баста,Сансара,2017-04-25,russian
3751,2019,radio,Emin,Невероятная,2018-06-15,russian
5940,2019,youtube,ЭGO,Ай,2019-07-15,russian
1322,2007,radio,Rihanna,Unfaithful,2006-09-14,english
6813,2024,youtube,Shukasha,Эрондондон,2024-03-10,russian
3512,2018,radio,Bazzi,Mine,2018-02-28,english
544,2003,radio,Моральный кодекс,Актриса по жизни,2003-12-02,russian
1950,2010,radio,Basic Element,Touch You Right Now,2009-03-05,english
4596,2024,radio,Kygo,Whatever,2024-01-25,english
